# PACT selective correction
Frozen-LLM CPU gate learning. See docs/selective_correction_colab.md. All execution switches default false; Run All does not launch inference.


In [ ]:
RUN_SOURCE_SMOKE = False
RUN_FIT_TUNING = False
RUN_CPU_FITS = False
RUN_CORE = False
RUN_TRANSFER = False

from pathlib import Path
import os, re, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/Soqoro/Pact.git'
GIT_REF = 'PASTE_THE_FULL_REVIEWED_COMMIT_SHA'
assert re.fullmatch(r'[0-9a-fA-F]{40}', GIT_REF), 'Set the pushed full SHA'
SCRATCH = Path('/content/pact-selective')
CHECKOUT = SCRATCH / 'checkout'
RUN = SCRATCH / 'runs' / 'pact-selective-correction-001'
PERSISTENT = Path('/content/drive/MyDrive/PACT/selective-correction/pact-selective-correction-001')
SCRATCH.mkdir(parents=True, exist_ok=True)
if not CHECKOUT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(CHECKOUT)], check=True)
assert not subprocess.check_output(['git', '-C', str(CHECKOUT), 'status', '--porcelain'], text=True).strip()
assert subprocess.check_output(['git', '-C', str(CHECKOUT), 'remote', 'get-url', 'origin'], text=True).strip() == REPO_URL
subprocess.run(['git', '-C', str(CHECKOUT), 'fetch', 'origin', GIT_REF], check=True)
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', GIT_REF], check=True)
assert subprocess.check_output(['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip().lower() == GIT_REF.lower()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(CHECKOUT)+'[model,gate]'], check=True)
os.environ['PYTHONPATH'] = str(CHECKOUT / 'src')
os.chdir(CHECKOUT)

In [ ]:
import json, time
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass  # If gated model access fails, stop and resolve account access.

def study(stage, *options):
    command = [sys.executable, '-u', '-m', 'pact', 'selective-correction', stage,
               '--run-dir', str(RUN), '--persistent', str(PERSISTENT),
               '--cache-dir', str(SCRATCH / 'cache'), *map(str, options)]
    log = SCRATCH / f'{stage}-{time.time_ns()}.log'
    print('Starting:', stage, 'Log:', log, flush=True)
    with log.open('w') as output:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            print(line, end='', flush=True)
            output.write(line); output.flush()
        code = proc.wait()
    if code:
        raise RuntimeError(f'{stage} stopped ({code}); preserve scratch and inspect {log}. No automatic retry.')

def read(path):
    return json.loads(Path(path).read_text())

In [ ]:
if not (RUN / 'plan.json').exists():
    if (PERSISTENT / 'snapshots').exists():
        study('restore')
    else:
        EXPOSURE = Path('/content/drive/MyDrive/PACT/selective-correction/prior-exposure.json')
        assert EXPOSURE.is_file(), 'Reviewed prior/locked exposure inventory missing; see prerequisites'
        SOURCE = SCRATCH / 'prepared-source.json'
        if not SOURCE.exists():
            study('prepare-source', '--source', SOURCE, '--exposure', EXPOSURE, '--download')
        study('plan', '--source', SOURCE)
PLAN_HASH = read(RUN / 'state.json')['plan_hash']
study('inspect')
print('PLAN_HASH:', PLAN_HASH)

In [ ]:
if RUN_SOURCE_SMOKE:
    study('preflight')
    study('collect-source', '--split', 'fit', '--smoke', '--execute', '--plan-hash', PLAN_HASH)
    study('report')
    study('export')
else:
    print("Stage disabled: RUN_SOURCE_SMOKE")

In [ ]:
if RUN_FIT_TUNING:
    SPLIT = 'fit'  # Then run this cell again with 'threshold_tuning'.
    study('collect-source', '--split', SPLIT, '--execute', '--plan-hash', PLAN_HASH)
    study('acquire-rewrites', '--split', SPLIT, '--execute', '--plan-hash', PLAN_HASH)
    study('build-check-bank', '--split', SPLIT, '--execute', '--plan-hash', PLAN_HASH)
    study('export')
else:
    print("Stage disabled: RUN_FIT_TUNING")

In [ ]:
if RUN_CPU_FITS:
    study('fit-gates')
    study('tune-thresholds')
    study('freeze-policies')
    study('export')
    print(read(RUN / 'fit_diagnostics.json'))
    print({arm: value['threshold'] for arm, value in read(RUN / 'threshold_selection.json').items()})
else:
    print("Stage disabled: RUN_CPU_FITS")

In [ ]:
if RUN_CORE:
    SPLIT = 'source_development_holdout'
    study('collect-source', '--split', SPLIT, '--execute', '--plan-hash', PLAN_HASH)
    study('acquire-rewrites', '--split', SPLIT, '--execute', '--plan-hash', PLAN_HASH)
    study('evaluate-core', '--execute', '--plan-hash', PLAN_HASH)
    study('report')
    study('export')
else:
    print("Stage disabled: RUN_CORE")

In [ ]:
if RUN_TRANSFER:
    from pact.storage import storage_operation
    PARENT = SCRATCH / 'adjudication-parent.zip'
    PARENT_SHA = '2abda348768e65ec8289f51eee48f8f5653ee0b36d30aff8c9093a7dc9c2ae8f'
    DRIVE_PARENT = Path('/content/drive/MyDrive/PACT/adjudication/pact-adjudication-diagnostic-001/bundles/pact-adjudication-diagnostic-001-PRIVATE-1791903639856421059.zip')
    if not PARENT.exists():
        print(storage_operation('bundle-restore', DRIVE_PARENT, PARENT, sha256=PARENT_SHA, timeout_seconds=600))
    study('import-transfer', '--bundle', PARENT, '--sha256', PARENT_SHA)
    study('evaluate-transfer', '--execute', '--plan-hash', PLAN_HASH)
    study('report')
    study('export')
else:
    print("Stage disabled: RUN_TRANSFER")